In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%pip -q install -U trl transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.2 MB/s eta 0:00:00


In [ ]:
%pip install flash-attn --no-build-isolation   # for A100 when we use flash-attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 112.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /content/Active-Reading--Pattern-Recognition

os.getcwd()

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Cloning into 'Active-Reading--Pattern-Recognition'...
remote: Enumerating objects: 5387, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 5387 (delta 133), reused 107 (delta 107), pack-reused 5241 (from 1)
Receiving objects: 100% (5387/5387), 181.14 MiB | 31.33 MiB/s, done.
Resolving deltas: 100% (1194/1194), done.
Updating files: 100% (4167/4167), done.
Error downloading object: Fine-tuning-configurations/final_qlora_adapter_paraphr_financeb/adapter_model.safetensors (a50ed13): Smudge error: Error downloading Fine-tuning-configurations/final_qlora_adapter_paraphr_financeb/adapter_model.safetensors (a50ed136d086832bafdb9106106f84b2999c50b9112404aed2314a5dd36704c9): batch response: This repository exceeded its LFS budget. The account responsible for the budget should increase it to restore access.

Errors logged to /content/Active-Reading--Pattern-Recognition/.git/lfs/logs/20260202T162958.997944979.log
Use `git lfs logs

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from itertools import chain
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from pathlib import Path
from datasets import concatenate_datasets


RUN_DIR = "/content/drive/MyDrive/qa_finetune/qlora_runs_mixture/qwen4b_qa_checkpoints"
SAVE_DIR = "/content/drive/MyDrive/qa_finetune/qlora_runs_mixture/final_qlora_adapter"


# --- Config ---
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
ORIGINAL_DATA_PATH =  "/content/Active-Reading--Pattern-Recognition/Finetune_Datasets/simplewiki/original_mixin.jsonl"
DATA_PATH = "/content/Active-Reading--Pattern-Recognition/Finetune_Datasets/simplewiki/mixture_dataset.jsonl"
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4


# --- Dataset ---
mix_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
original_dataset = load_dataset("json", data_files=ORIGINAL_DATA_PATH, split="train")



dataset = concatenate_datasets([mix_dataset, original_dataset])


# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



def to_text(ex):
    if "question" in ex and "answer" in ex and ex["question"] and ex["answer"]:
        q = ex["question"].strip()
        a = ex["answer"].strip()
        txt = q + "\n" + a
    elif "active_reading" in ex and ex["active_reading"]:
        txt = ex["active_reading"].strip()
    else:
        txt = ex.get("text", "")
        txt = txt.strip()
    return {"text": txt + tokenizer.eos_token}


dataset = dataset.map(to_text, remove_columns=dataset.column_names)



# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.float16, # for T4-GPU
    bnb_4bit_compute_dtype=torch.bfloat16, # for A100
    bnb_4bit_use_double_quant=True,
)

# --- Load Model (QLoRA) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2", # for A100
)

# --- Prep for k-bit training ---
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# --- LoRA ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH

    result = {
        k: [t[i:i + MAX_SEQ_LENGTH] for i in range(0, total_length, MAX_SEQ_LENGTH)]
        for k, t in concatenated.items()
    }
    return result


def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )
    return tokenized


tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=2
)


lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    num_proc=2
)


# --- Training Args ---
training_args = TrainingArguments(
    output_dir=RUN_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LEARNING_RATE,
    max_steps=400,
    # fp16=True, # for T4-GPU
    bf16=True, # for A100
    tf32=True, # for A100
    logging_steps=10,
    save_steps=40,
    save_total_limit=2,
    report_to="none",
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print(" Starting QLoRA repetition fine-tuning...")
#trainer.train(resume_from_checkpoint=True)
trainer.train()

print(" Saving adapter...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

 Starting QLoRA repetition fine-tuning...


Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss
10,1.834250
20,1.776467
30,1.697046
40,1.635182
50,1.651752
60,1.690214
70,1.636644
80,1.556830
90,1.593904
100,1.623816


 Saving adapter...


('/content/drive/MyDrive/qa_finetune/qlora_runs_mixture/final_qlora_adapter/tokenizer_config.json',
 '/content/drive/MyDrive/qa_finetune/qlora_runs_mixture/final_qlora_adapter/chat_template.jinja',
 '/content/drive/MyDrive/qa_finetune/qlora_runs_mixture/final_qlora_adapter/tokenizer.json')